# Late-tail spatial map

This notebook is the slower companion to `reports/late-tail-spatial-map.md`.

The narrow question is the useful one: once a start lands in the slow tail, is it part of a thin basin filament or part of a thicker center trap?


## Load the tiled map

The CSV sidecar stores one row per tile. Each tile reports the share of starts that either need at least 20 Newton steps or fail the 40-step cutoff.


In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

rows = []
with (Path('..') / 'art' / 'late-tail-spatial-map.csv').open() as handle:
    for row in csv.DictReader(handle):
        row['power'] = int(row['power'])
        row['tile_x'] = int(row['tile_x'])
        row['tile_y'] = int(row['tile_y'])
        row['sample_count'] = int(row['sample_count'])
        for key in ['x_min', 'x_max', 'y_min', 'y_max', 'mean_iterations', 'late_fraction', 'unresolved_fraction']:
            row[key] = float(row[key])
        rows.append(row)

by_power = defaultdict(list)
for row in rows:
    by_power[row['power']].append(row)
sorted(by_power)


## Hottest tile per power

This reproduces the quickest public summary from the report: where is the hottest local patch, and how saturated is it?


In [ ]:
summary = []
for power in sorted(by_power):
    hottest = max(by_power[power], key=lambda row: row['late_fraction'])
    grid_late = sum(row['sample_count'] * row['late_fraction'] for row in by_power[power]) / sum(row['sample_count'] for row in by_power[power])
    center_rows = sorted(by_power[power], key=lambda row: abs((row['x_min'] + row['x_max']) / 2) + abs((row['y_min'] + row['y_max']) / 2))[:4]
    center_late = sum(row['late_fraction'] for row in center_rows) / len(center_rows)
    summary.append({
        'power': power,
        'grid_late': round(grid_late, 3),
        'center_late': round(center_late, 3),
        'hot_tile': (round(hottest['x_min'], 2), round(hottest['x_max'], 2), round(hottest['y_min'], 2), round(hottest['y_max'], 2)),
        'hot_fraction': round(hottest['late_fraction'], 3),
    })
summary


The progression is the point.

- `z^3 - 1` still spends most of its late mass on thin filament regions
- the higher powers stop looking filament-first and start filling the center blocks almost completely

That is more specific than saying the inner radius bands get slower. The map shows *how* they get slower.


## Compare center mass against the whole grid

A simple ratio keeps the visual takeaway honest. If the center average is much larger than the grid average, the slow tail has condensed into a center trap instead of staying diffuse.


In [ ]:
[(row['power'], round(row['center_late'] / row['grid_late'], 2)) for row in summary if row['grid_late'] > 0]


## Takeaway

The earlier histogram and radius passes were still worth doing. They just hid one geometric fact that matters for reading the family: late starts change shape as the power rises.

1. low powers keep the slow tail mostly on narrow boundary filaments
2. higher powers turn the origin neighborhood into a broad finite-budget trap
3. so the hard part is not only that more points are slow — it is that the slow region stops being thin

That is why this sidecar earns a slot instead of just repainting the older figures.
